In [ ]:
"""
Phase 1: O*NET Task Classifier — PRODUCTION RUN
=================================================
Adapted from the validity-check draft. This version is locked down for the
production pass: mid-tier models only, single temperature, one run each.

    3 providers (mid-tier) x 1 temperature (0.2) x 1 run = 3 calls per task

Install:
    pip install anthropic openai google-genai pydantic pandas tenacity

Set env vars:
    export ANTHROPIC_API_KEY=""
    export OPENAI_API_KEY=""
    export GEMINI_API_KEY=""

Run:
    python production_classification_script.py
"""

!pip install anthropic openai google-genai pydantic pandas

In [2]:
import os
import json
import csv
import time
import re
from pathlib import Path

import anthropic
import openai
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
import pandas as pd

from tenacity import (
    retry,
    retry_if_exception,
    stop_after_attempt,
    wait_exponential,
)

os.environ["ANTHROPIC_API_KEY"] = ""
os.environ["OPENAI_API_KEY"] = ""
os.environ["GEMINI_API_KEY"] = ""

In [38]:
# ---------------------------------------------------------------------------
# 1. MODEL CONFIGURATION
# ---------------------------------------------------------------------------
# Production run uses MID_TIER only — flagship models are not called here.

MID_TIER = {
    "claude": "claude-sonnet-4-6",
    "openai": "gpt-5.4-mini",
    "gemini": "gemini-3.1-flash-lite",
}


# ---------------------------------------------------------------------------
# 2. SUBPROCESS DEFINITIONS, DECISION RULES, RESPONSE FORMAT, SYSTEM PROMPT
#    (carried over unchanged from the validated v3 prompt — no edits here)
# ---------------------------------------------------------------------------

SUBPROCESS_DEFINITIONS = """
1. Master Data Management — Establishing and maintaining the structural and 
   governance foundation of the financial architecture: the chart of accounts, 
   reporting entity hierarchies, cost and profit centre structures, 
   company-specific attributes, currency rate tables, system interfaces, and user 
   access rights. Provides the infrastructure on which all downstream subprocesses 
   depend, without itself processing period transactions.

2. Record — The ongoing accumulation and initial processing of financial 
   transactions during the fiscal period, per accepted accounting principles: 
   processing journal entries, posting transactions to subledgers, recording accruals, 
   and performing transaction-level reconciliations as activity occurs.

3. Close & Consolidation — The period-end procedures that finalise and aggregate 
   recorded transactions into group-level financial results: closing subledgers, executing 
   general ledger accounting, finalising accruals against the close deadline, reconciling GL 
   accounts and resolving variances, managing intercompany accounting and eliminations, 
   calculating tax liability (different from tax filings), and consolidating entity results into 
   preliminary group financial statements.

4. Report — Translating the closed and consolidated ledger into structured outputs 
   for stakeholders: producing internal financial management reports, external 
   financial statements, and tax filings, in order to meet the organisation's 
   internal and external disclosure obligations.

5. Analysis & Business Insight — Interrogating reported financial results to 
   generate interpretive, decision-useful intelligence: performing variance and 
   performance analysis, developing and maintaining KPIs, and providing insight 
   intended to influence management decisions, including post-close review and 
   identification of needed adjustments.

6. Planning & Budgeting — Translating analytical findings into forward-looking 
   financial direction: developing long-range strategic plans, annual budgets and 
   operating plans, rolling forecasts, product costing, and performance targets 
   that govern the next iteration of the cycle.
   
7. AMBIGUOUS — Use only when both of the following conditions are met:
  (a) The task is clearly within R2R scope — it constitutes financial recording, 
      consolidation, disclosure, analysis, or planning work (Rule 0 is satisfied).
  (b) Rules 1–4, applied in sequence, still do not resolve the task to a single 
      subprocess — because the task description genuinely and equally spans two or 
      more subprocesses with no resolvable priority between them.

AMBIGUOUS is not a hedge for difficult tasks. If the rules resolve the task, even 
with effort, do not use AMBIGUOUS. Reserve it for cases where the task description 
is constitutively split — where assigning a single label would misrepresent what 
the task actually does.

8. OUT_OF_SCOPE — Use when Rule 0 is not satisfied: the task does not itself perform 
R2R work, regardless of its organisational context or eventual purpose. 
"""

DECISION_RULES = """
Apply Rules 0 through 4 strictly in numerical order, one rule at a time. Fully
resolve or fully rule out each rule before moving to the next — do not jump
ahead to a later rule to justify a classification that an earlier rule would
have already determined differently. Begin every task with Rule 0.

Rule 0 — Before applying this rule, read the subprocess definitions carefully. 
  The scope of R2R in this study is determined exclusively by those definitions, 
  not simply general intuitions about what constitutes "financial" work. Do not exclude 
  a task on the grounds that it resembles operational, engineering, or non-accounting 
  work if the subprocess definitions would encompass it. A task is in scope if it constitutes 
  activity within any of the six subprocess definitions provided. If the task instead 
  consists of data-gathering, or cross-functional communication that precedes and informs 
  subprocess activity — but does not itself constitute such activity as defined — classify 
  as OUT_OF_SCOPE.
  
  Exception — Master Data Management carve-in: A task that maintains,
  establishes, or governs a structural element of the financial architecture
  (chart of accounts, entity hierarchy, cost/profit centres, currency tables,
  system access) IS constitutive R2R activity, even though it does not itself
  process a transaction, figure, report, or plan. Treat it as in-scope and
  proceed to Rule 3.

  Carve-out — Audit/assurance activity: A task describing audit, assurance, or
  internal control work — testing controls, verifying compliance, examining
  records or processes for accuracy as an independent check, assessing the
  adequacy of a control environment — is OUT_OF_SCOPE, even when it involves
  reviewing financial records or transactions. Its purpose is independent
  verification of work already performed, not the performance of the
  recording, consolidation, disclosure, analysis, or planning itself. (O*NET
  groups "Accountants and Auditors" under shared occupational codes; this
  carve-out separates the two distinct job functions.)

  Carve-out — Tax planning/strategy: A task describing tax planning, tax
  strategy, or general tax advisory (e.g., developing tax-minimisation
  strategies, advising on entity tax structuring) is OUT_OF_SCOPE. It does not
  itself produce a recorded entry, consolidated tax provision, filed
  disclosure, or budget. Distinguish from tax provision calculation (→ Close &
  Consolidation) and tax filing/return preparation (→ Report), which remain
  in scope. Distinguish from other financial planning and strategic activities, which
  remain in scope.

Rule 1 — Purpose of output (primary rule).
  Classify by why the output is being produced, not by the activity that
  produces it. A task that prepares reports for management analysis is
  Analysis & Business Insight, not Report, because the purpose is decision
  support, not disclosure.

  Note: This rule routes a task to Analysis & Business Insight only when the
  analysis interrogates the organisation's own recorded financial or
  accounting data (e.g., variance to budget, financial KPI trend, post-close
  review). Analysis of external information — market research, competitor
  benchmarking, macroeconomic or industry forecasting — does not qualify, even
  if the task is described as "financial analysis"; such tasks fail Rule 0 and
  are OUT_OF_SCOPE.

Rule 2 — Output type (fallback when Rule 1 leaves no clear purpose).
  Classify by the type of artefact produced:
    - A recorded or reconciled entry → Record
    - A consolidated group figure → Close & Consolidation
    - A disclosure document or regulatory filing → Report
    - A budget, forecast, or performance target → Planning & Budgeting

Rule 3 — Maintenance vs. execution (Master Data Management boundary).
  A task framed as maintaining, establishing, or governing a structural
  element (chart of accounts, entity hierarchy, cost/profit centres, currency
  tables, system access) → Master Data Management, regardless of which
  subprocess uses that structure. A task that uses such structure to process
  a period's transactions → the relevant operational subprocess per Rules 1–2.

Rule 4 — Transaction-level vs. period-end (Record vs. Close &
Consolidation).
  Is the task processing an individual transaction, or closing out a period?
  Transaction-level reconciliations and adjustments → Record.
  Account-balance or period-end reconciliations, adjustments, and reviews →
  Close & Consolidation.

Rule 5 — Single-label enforcement.
  Apply Rules 0–4 in sequence to arrive at exactly one label, even where a
  task description spans multiple subprocesses. Multi-subprocess phrasing
  alone is not grounds for AMBIGUOUS. Reserve AMBIGUOUS only for genuine
  unresolved contestation after all rules are exhausted, where both purpose and output 
  rules cannot resolve which subprocess to put the task in.
"""

ALLOWED_LABELS = [
    "Master Data Management",
    "Record",
    "Close & Consolidation",
    "Report",
    "Analysis & Business Insight",
    "Planning & Budgeting",
    "AMBIGUOUS",
    "OUT_OF_SCOPE",
]

RESPONSE_FORMAT = """
RESPONSE FORMAT:
Return raw JSON only — no other text before or after, and no markdown code
fences (do not wrap the JSON in ```json ... ```). Your response must start
with the character { and end with the character }.

{
  "reasoning": "<in 1 to 2 sentences, strictly maximum 60 words: name the determinative rule (e.g. 'Rule 0 exception' or 'Rule 4') and the fact that triggered it — do not restate rule text or walk through every rule considered>",
  "classification": "<exactly one of the allowed labels>"
}

Example (illustrative of format only — not a subprocess example):
Input task: "Liaise with business units to collect headcount data for inclusion in 
the annual workforce plan."
Output:
{
  "reasoning": "Data-gathering that feeds planning but produces no record itself; Rule 0, out of scope.",
  "classification": "OUT_OF_SCOPE"
}
"""

SYSTEM_PROMPT = f"""You are an expert in Record-to-Report (R2R) process classification. \
Your task is to assign each O*NET task statement to exactly one of the six R2R \
subprocesses below, or flag it as AMBIGUOUS or OUT_OF_SCOPE.

SUBPROCESS DEFINITIONS:
{SUBPROCESS_DEFINITIONS}
DECISION RULES:
{DECISION_RULES}
ALLOWED LABELS — use exactly as written:
{chr(10).join(f'  - {label}' for label in ALLOWED_LABELS)}

{RESPONSE_FORMAT}"""


def build_user_message(task_id: str, task_statement: str) -> str:
    return (
        f"Task ID: {task_id}\n"
        f"Task Statement: {task_statement}\n\n"
        f"Classify this task."
    )


# ---------------------------------------------------------------------------
# 3. PYDANTIC SCHEMA  (for OpenAI + Gemini native structured output)
# ---------------------------------------------------------------------------
class TaskClassification(BaseModel):
    reasoning:      str = Field(description="Step-by-step chain-of-thought.")
    classification: str = Field(description=f"One of: {', '.join(ALLOWED_LABELS)}")


# ---------------------------------------------------------------------------
# 4. THREE PROVIDER FUNCTIONS  — same inputs, same output shape
# ---------------------------------------------------------------------------
def is_transient_error(exc: BaseException) -> bool:
    """Shared predicate so one retry decorator works across all three SDKs
    (anthropic / openai / google-genai), each of which raises a different
    exception class on a 503. Checks status_code where the SDK exposes one,
    falls back to matching common transient-error text."""
    status = getattr(exc, "status_code", None)
    if status in {429, 500, 502, 503, 504, 529}:
        return True
    msg = str(exc).lower()
    return any(
        s in msg
        for s in ("503", "529", "overloaded", "rate limit", "rate_limit", "unavailable", "internal server error")
    )


def _parse_json(raw: str) -> tuple[str, str]:
    """Parse JSON response; return (classification, reasoning).

    Strips markdown code fences before parsing — Claude has been observed
    wrapping its JSON in ```json ... ``` blocks despite the system prompt
    instructing otherwise. Only call_claude routes through this function;
    call_openai and call_gemini use native structured output (.parse() /
    response_schema) and never hit raw-string parsing.
    """
    cleaned = raw.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        parsed = json.loads(cleaned)
        return parsed.get("classification", "PARSE_ERROR"), parsed.get("reasoning", "")
    except json.JSONDecodeError:
        return "PARSE_ERROR", raw


@retry(
    retry=retry_if_exception(is_transient_error),
    wait=wait_exponential(multiplier=2, min=2, max=8),
    stop=stop_after_attempt(4),
    reraise=True,
)
def call_claude(task_id: str, task_statement: str, model: str, temperature: float) -> dict:
    client   = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY
    response = client.messages.create(
        model=model,
        max_tokens=512,
        temperature=temperature,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": build_user_message(task_id, task_statement)}],
    )
    classification, reasoning = _parse_json(response.content[0].text.strip())
    return {
        "classification": classification,
        "reasoning":      reasoning,
        "input_tokens":   response.usage.input_tokens,
        "output_tokens":  response.usage.output_tokens,
    }


@retry(
    retry=retry_if_exception(is_transient_error),
    wait=wait_exponential(multiplier=2, min=2, max=8),
    stop=stop_after_attempt(4),
    reraise=True,
)
def call_openai(task_id: str, task_statement: str, model: str, temperature: float) -> dict:
    client   = openai.OpenAI()         # reads OPENAI_API_KEY
    response = client.beta.chat.completions.parse(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_message(task_id, task_statement)},
        ],
        response_format=TaskClassification,
    )
    result = response.choices[0].message.parsed
    return {
        "classification": result.classification,
        "reasoning":      result.reasoning,
        "input_tokens":   response.usage.prompt_tokens,
        "output_tokens":  response.usage.completion_tokens,
    }


@retry(
    retry=retry_if_exception(is_transient_error),
    wait=wait_exponential(multiplier=2, min=2, max=8),
    stop=stop_after_attempt(4),
    reraise=True,
)
def call_gemini(task_id: str, task_statement: str, model: str, temperature: float) -> dict:
    client   = genai.Client()          # reads GEMINI_API_KEY
    response = client.models.generate_content(
        model=model,
        contents=build_user_message(task_id, task_statement),
        config=types.GenerateContentConfig(
            temperature=temperature,
            system_instruction=SYSTEM_PROMPT,
            response_mime_type="application/json",
            response_schema=TaskClassification,
        ),
    )
    parsed = json.loads(response.text)
    return {
        "classification": parsed.get("classification", "PARSE_ERROR"),
        "reasoning":      parsed.get("reasoning", ""),
        "input_tokens":   response.usage_metadata.prompt_token_count,
        "output_tokens":  response.usage_metadata.candidates_token_count,
    }


# ---------------------------------------------------------------------------
# 5. DISPATCHER  — one function to call any provider
# ---------------------------------------------------------------------------
PROVIDER_FNS = {
    "claude": call_claude,
    "openai": call_openai,
    "gemini": call_gemini,
}


def classify_one(
    provider:       str,
    task_id:        str,
    task_statement: str,
    model:          str,
    temperature:    float,
    run_id:         int,
) -> dict:
    """Single classification call. Returns a flat dict ready for CSV."""
    result = PROVIDER_FNS[provider](task_id, task_statement, model, temperature)
    return {
        "task_id":        task_id,
        "task_statement": task_statement,
        "provider":       provider,
        "model":          model,
        "temperature":    temperature,
        "run_id":         run_id,
        **result,          # classification, reasoning, input_tokens, output_tokens
    }

In [39]:
# ---------------------------------------------------------------------------
# 6. SINGLE-TASK SMOKE TEST  — run this first to confirm everything works
# ---------------------------------------------------------------------------
def test_single_task():
    """
    Run one task through all three mid-tier models at temperature=0.2.
    Use this to verify your API keys and prompt before the full production run.
    """
    task_id        = "3394"
    task_statement = (
        "Direct or coordinate an organization's financial or budget activities to fund operations, maximize investments, or increase efficiency."
    )

    print("=" * 60)
    print("SINGLE-TASK SMOKE TEST  (mid-tier, temperature=0.2)")
    print("=" * 60)

    for provider, model in MID_TIER.items():
        print(f"\n[{provider.upper()}]  {model}")
        try:
            result = classify_one(
                provider=provider,
                task_id=task_id,
                task_statement=task_statement,
                model=model,
                temperature=0.2,
                run_id=1,
            )
            print(f"  Classification : {result['classification']}")
            print(f"  Reasoning      : {result['reasoning']}")
            print(f"  Tokens used    : {result['input_tokens']} in / {result['output_tokens']} out")
        except Exception as e:
            print(f"  ERROR: {e}")

test_single_task()

SINGLE-TASK SMOKE TEST  (mid-tier, temperature=0.2)

[CLAUDE]  claude-sonnet-4-6
  Classification : Planning & Budgeting
  Reasoning      : Rule 1 determines this: the purpose spans both budgeting/planning (fund operations, performance targets) and broader financial direction. However, 'budget activities' and 'maximize investments/increase efficiency' point primarily to forward-looking financial direction, making Planning & Budgeting the dominant purpose.
  Tokens used    : 2244 in / 89 out

[OPENAI]  gpt-5.4-mini
  Classification : OUT_OF_SCOPE
  Reasoning      : Rule 0 carve-out: this is tax/financial strategy and management of funding/investment decisions, not recording, closing, reporting, analysis of results, or budgeting output itself. It describes directing financial activities to optimize operations and investments, which is out of R2R scope.
  Tokens used    : 2112 in / 74 out

[GEMINI]  gemini-3.1-flash-lite
  Classification : OUT_OF_SCOPE
  Reasoning      : The task describe

In [40]:
# ---------------------------------------------------------------------------
# 7. PRODUCTION RUN — mid-tier only, temperature = 0.2, 1 run per model
# ---------------------------------------------------------------------------
# Locked-down config for the production pass (no flagship cross-check here,
# no temperature sweep, no repeated runs — that all happened in the
# validity-check stage). This is the final, single-pass classification grid.

TEMPERATURE   = 0.2
RUNS_PER_TEMP = 1
OUTPUT_DIR    = Path("production outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Total per task: 3 mid-tier providers x 1 temperature x 1 run = 3 calls

FIELDNAMES = [
    "task_id", "task_statement", "tier", "provider", "model",
    "temperature", "run_id", "classification", "reasoning",
    "input_tokens", "output_tokens",
]


def run_production(
    tasks:      list[dict],
    output_csv: str = "production_run.csv",
    sleep:      float = 0.5,
) -> pd.DataFrame:
    """
    Production classification grid per task:
        3 mid-tier providers x 1 temperature (0.2) x 1 run = 3 calls

    Writes every result to CSV immediately — safe to inspect mid-run.
    Output goes to OUTPUT_DIR / output_csv, i.e. "production outputs/production_run.csv".
    """
    csv_path = OUTPUT_DIR / output_csv
    results  = []

    total = len(tasks) * len(MID_TIER) * RUNS_PER_TEMP

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
        done = 0

        for task_dict in tasks:
            for provider, model in MID_TIER.items():
                for run in range(1, RUNS_PER_TEMP + 1):
                    done += 1
                    print(
                        f"[{done}/{total}]  "
                        f"mid | {provider} | temp={TEMPERATURE} | run={run} | "
                        f"task={task_dict['task_id']}"
                    )
                    try:
                        row = classify_one(
                            provider=provider,
                            task_id=task_dict["task_id"],
                            task_statement=task_dict["task"],
                            model=model,
                            temperature=TEMPERATURE,
                            run_id=run,
                        )
                    except Exception as e:
                        print(f"  ERROR: {e}")
                        row = {
                            "task_id":        task_dict["task_id"],
                            "task_statement": task_dict["task"],
                            "provider":       provider,
                            "model":          model,
                            "temperature":    TEMPERATURE,
                            "run_id":         run,
                            "classification": "API_ERROR",
                            "reasoning":      str(e),
                            "input_tokens":   0,
                            "output_tokens":  0,
                        }

                    row["tier"] = "mid"
                    results.append(row)
                    writer.writerow(row)
                    f.flush()
                    time.sleep(sleep)

    df = pd.DataFrame(results)
    print(f"\nProduction run complete. {len(df)} rows saved -> {csv_path}")
    return df

In [41]:
# ---------------------------------------------------------------------------
# 8. CSV LOADER
# ---------------------------------------------------------------------------
def load_tasks_from_csv(filepath: str) -> list[dict]:
    """
    Reads your O*NET CSV and returns a list of {"task_id", "task"} dicts.
    Expects columns named exactly: "Task ID" and "Task"
    (as they appear in the O*NET task_statements download).
    """
    df = pd.read_csv(filepath, dtype={"Task ID": str})

    missing = [c for c in ["Task ID", "Task"] if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing columns in CSV: {missing}\n"
            f"Found columns: {list(df.columns)}"
        )

    tasks = [
        {"task_id": row["Task ID"].strip(), "task": row["Task"].strip()}
        for _, row in df.iterrows()
        if pd.notna(row["Task ID"]) and pd.notna(row["Task"])
    ]
    print(f"Loaded {len(tasks)} tasks from {filepath}")
    return tasks

In [43]:
# ---------------------------------------------------------------------------
# 9. ENTRY POINT
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    tasks = load_tasks_from_csv("Tasks_Only.csv")
    run_production(tasks, output_csv="production_run.csv")

Loaded 212 tasks from Tasks_Only.csv
[1/636]  mid | claude | temp=0.2 | run=1 | task=8823
[2/636]  mid | openai | temp=0.2 | run=1 | task=8823
[3/636]  mid | gemini | temp=0.2 | run=1 | task=8823
[4/636]  mid | claude | temp=0.2 | run=1 | task=8824
[5/636]  mid | openai | temp=0.2 | run=1 | task=8824
[6/636]  mid | gemini | temp=0.2 | run=1 | task=8824
[7/636]  mid | claude | temp=0.2 | run=1 | task=8827
[8/636]  mid | openai | temp=0.2 | run=1 | task=8827
[9/636]  mid | gemini | temp=0.2 | run=1 | task=8827
[10/636]  mid | claude | temp=0.2 | run=1 | task=8826
[11/636]  mid | openai | temp=0.2 | run=1 | task=8826
[12/636]  mid | gemini | temp=0.2 | run=1 | task=8826
[13/636]  mid | claude | temp=0.2 | run=1 | task=8834
[14/636]  mid | openai | temp=0.2 | run=1 | task=8834
[15/636]  mid | gemini | temp=0.2 | run=1 | task=8834
[16/636]  mid | claude | temp=0.2 | run=1 | task=8836
[17/636]  mid | openai | temp=0.2 | run=1 | task=8836
[18/636]  mid | gemini | temp=0.2 | run=1 | task=8836
